# Notebook 6 - Trade Settlement Risk Prediction
Predict settlement risk using a labeled dataset.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score

# Replace with your settlement dataset
df = pd.read_csv("data/trade_settlement/trade_settlement.csv")

# Expected target column:
# Settlement_Risk (0=No Risk, 1=Risk)
target="Settlement_Risk"

# Select numeric features automatically
features=df.select_dtypes(include="number").columns.drop(target)
X=df[features]
y=df[target]

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y)


## Train Random Forest

In [ ]:
model=Pipeline([
("imputer",SimpleImputer(strategy="median")),
("rf",RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"))
])

model.fit(X_train,y_train)
pred=model.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))
print(classification_report(y_test,pred))
print(confusion_matrix(y_test,pred))


## Feature Importance

In [ ]:
rf=model.named_steps["rf"]
importance=pd.DataFrame({
    "Feature":features,
    "Importance":rf.feature_importances_
}).sort_values("Importance",ascending=False)

importance


## Score All Trades

In [ ]:
df["Predicted_Risk"]=model.predict(X)
df["Risk_Probability"]=model.predict_proba(X)[:,1]

high_risk=df[df["Predicted_Risk"]==1]
high_risk.head()


## Save Outputs

In [ ]:
importance.to_csv("processed/settlement_feature_importance.csv",index=False)
df.to_csv("processed/trade_settlement_predictions.csv",index=False)
print("Outputs saved.")


## Suggested Features

- Trade Value
- Quantity
- Instrument Type
- Exchange
- Broker
- Counterparty
- Settlement Cycle (T+1/T+2)
- Trade Date
- Settlement Date
- Currency
- Failed Settlement History
- Country
- Liquidity Score
- Volatility
